# Text Feature Engineering Assignment
**Dataset:** 1000 real user reviews (500 Google Play Store + 500 Steam)  
**Tasks:** Preprocessing → Vocabulary → OHE / BoW / TF-IDF → Analysis → Sentiment Classification

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('All imports OK')

## Load Dataset

In [ ]:
df = pd.read_csv('all_reviews.csv')
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nSource distribution:')
print(df['source'].value_counts())
df.head()

In [ ]:
# Drop rows with missing review text
df = df.dropna(subset=['review_text']).reset_index(drop=True)
print('After dropping nulls:', df.shape)
print('\nSample reviews:')
for i, row in df.sample(3, random_state=42).iterrows():
    print(f"[{row['source']}] {row['review_text'][:120]}...\n")

---
## Task 1: Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text: str, remove_stopwords: bool = True, lemmatize: bool = True) -> str:
    """Full preprocessing pipeline: lowercase → punctuation removal → tokenize → stopwords → lemmatize."""
    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 3: Tokenize
    tokens = word_tokenize(text)

    # Step 4 (optional): Remove stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]

    # Step 5 (optional): Lemmatize
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

# Apply preprocessing
df['clean_text'] = df['review_text'].apply(preprocess)

# Show before / after
print('=== Before preprocessing ===')
print(df['review_text'].iloc[0][:200])
print('\n=== After preprocessing ===')
print(df['clean_text'].iloc[0][:200])

In [ ]:
# Review length distribution
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['word_count'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Word Count Distribution (after preprocessing)')
axes[0].set_xlabel('Words per review')
axes[0].set_ylabel('Count')

df.groupby('source')['word_count'].mean().plot(kind='bar', ax=axes[1], color=['#2196F3','#FF5722'])
axes[1].set_title('Avg Words per Review by Source')
axes[1].set_ylabel('Avg word count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()
print(df['word_count'].describe())

---
## Task 2: Vocabulary Creation

In [ ]:
# Build vocabulary manually
all_tokens = [token for text in df['clean_text'] for token in text.split()]
vocab_counter = Counter(all_tokens)
vocabulary = sorted(vocab_counter.keys())

print(f'Total tokens (with repetition) : {len(all_tokens):,}')
print(f'Vocabulary size (unique words)  : {len(vocabulary):,}')
print(f'\nTop 20 most frequent words:')
for word, count in vocab_counter.most_common(20):
    print(f'  {word:<20} {count}')

In [ ]:
# Visualize top 20 words
top_words = vocab_counter.most_common(20)
words, counts = zip(*top_words)

plt.figure(figsize=(12, 5))
plt.bar(words, counts, color='steelblue', edgecolor='white')
plt.title('Top 20 Most Frequent Words')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Task 3: Feature Engineering
### 3a — One Hot Encoding (document-level)

In [ ]:
# OHE: each document represented as a binary vector
# 1 = word present in document, 0 = absent (ignores frequency)
ohe_vectorizer = CountVectorizer(binary=True, max_features=500)
ohe_matrix = ohe_vectorizer.fit_transform(df['clean_text'])

print('OHE Matrix shape :', ohe_matrix.shape)
print('Data type        :', ohe_matrix.dtype)
print('Non-zero values  :', ohe_matrix.nnz)
print('\nSample OHE vector (first review, first 20 features):')
print(ohe_matrix[0, :20].toarray())
print('Feature names    :', ohe_vectorizer.get_feature_names_out()[:20].tolist())

### 3b — Bag of Words (CountVectorizer)

In [ ]:
bow_vectorizer = CountVectorizer(max_features=5000)
bow_matrix = bow_vectorizer.fit_transform(df['clean_text'])

print('BoW Matrix shape :', bow_matrix.shape)
print('Data type        :', bow_matrix.dtype)
print('Non-zero values  :', bow_matrix.nnz)
print('\nSample BoW vector (first review, first 20 features):')
print(bow_matrix[0, :20].toarray())
print('Feature names    :', bow_vectorizer.get_feature_names_out()[:20].tolist())

### 3c — TF-IDF (TfidfVectorizer)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_text'])

print('TF-IDF Matrix shape :', tfidf_matrix.shape)
print('Data type           :', tfidf_matrix.dtype)
print('Non-zero values     :', tfidf_matrix.nnz)
print('\nSample TF-IDF vector (first review, first 20 features):')
print(np.round(tfidf_matrix[0, :20].toarray(), 4))
print('Feature names       :', tfidf_vectorizer.get_feature_names_out()[:20].tolist())

---
## Task 4: Comparison Analysis

In [ ]:
comparison = pd.DataFrame({
    'Method': ['One Hot Encoding', 'Bag of Words', 'TF-IDF'],
    'Matrix Shape': [str(ohe_matrix.shape), str(bow_matrix.shape), str(tfidf_matrix.shape)],
    'Values': ['Binary (0/1)', 'Integer counts', 'Float (0–1)'],
    'Captures Frequency': ['No', 'Yes', 'Yes (weighted)'],
    'Penalises Common Words': ['No', 'No', 'Yes (IDF)'],
    'Use Case': ['Presence/Absence', 'Word frequency', 'Importance ranking'],
})
print(comparison.to_string(index=False))

In [ ]:
# Top TF-IDF words — words that are most DISTINCTIVE (not just frequent)
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_tfidf_idx = mean_tfidf.argsort()[-20:][::-1]

top_tfidf_words = [(feature_names[i], round(mean_tfidf[i], 4)) for i in top_tfidf_idx]
print('Top 20 words by mean TF-IDF score (most distinctive across corpus):')
for word, score in top_tfidf_words:
    print(f'  {word:<20} {score}')

In [ ]:
# Compare: BoW rank vs TF-IDF rank for same word
bow_features = bow_vectorizer.get_feature_names_out()
bow_sums = np.asarray(bow_matrix.sum(axis=0)).flatten()
top_bow = dict(zip(bow_features, bow_sums))

common_words = ['game', 'good', 'play', 'great', 'like', 'get', 'app', 'time']
print(f"{'Word':<15} {'BoW Count':>12} {'Mean TF-IDF':>14}  Note")
print('-' * 60)
for w in common_words:
    bow_val = top_bow.get(w, 0)
    tfidf_val = mean_tfidf[np.where(feature_names == w)[0][0]] if w in feature_names else 0
    note = 'high freq → low TF-IDF' if bow_val > 500 and tfidf_val < 0.02 else ''
    print(f"{w:<15} {int(bow_val):>12,} {tfidf_val:>14.4f}  {note}")

**Why common words get lower TF-IDF weight:**  
TF-IDF = TF × IDF. IDF = log(N / df) where df = number of documents containing the word.  
Words like *good*, *game*, *app* appear in almost every review → df ≈ N → IDF ≈ 0 → low TF-IDF.  
Rare, distinctive words (product names, specific complaints) get high IDF → high TF-IDF.

---
## Task 5: Sparse Matrix Analysis

In [ ]:
def sparsity(matrix):
    total = matrix.shape[0] * matrix.shape[1]
    nonzero = matrix.nnz
    return (1 - nonzero / total) * 100

for name, mat in [('OHE', ohe_matrix), ('BoW', bow_matrix), ('TF-IDF', tfidf_matrix)]:
    rows, cols = mat.shape
    total_elements = rows * cols
    sp = sparsity(mat)
    dense_mb = (total_elements * 8) / (1024**2)      # float64 dense
    sparse_mb = (mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes) / (1024**2)
    print(f'{name}')
    print(f'  Shape           : {mat.shape}')
    print(f'  Total elements  : {total_elements:,}')
    print(f'  Non-zero        : {mat.nnz:,}')
    print(f'  Sparsity        : {sp:.2f}%')
    print(f'  Dense size (MB) : {dense_mb:.1f} MB')
    print(f'  Sparse size (MB): {sparse_mb:.2f} MB')
    print()

**Why sparse matrices are inefficient for large-scale systems:**  
- With 1M documents and a 50K vocabulary, a dense BoW matrix = **1M × 50K × 8 bytes ≈ 400 GB**  
- 99%+ of values are zero — storing them wastes memory and slows computation  
- Solutions: use `scipy.sparse`, dimensionality reduction (SVD/PCA), or switch to dense embeddings (Word2Vec, BERT)

---
## Task 6: Real-world Questions

### Q1 — Why does Bag of Words fail at semantic meaning?

BoW treats each word as an independent token with no relationship to other words.  
- "*The game is amazing*" and "*The game is terrible*" have nearly identical BoW vectors  
- Synonyms: "*good*" and "*excellent*" are completely different dimensions in BoW  
- Word order ignored: "*dog bites man*" == "*man bites dog*" in BoW  

Modern fix: **word embeddings** (Word2Vec, GloVe, BERT) capture semantic similarity.

In [ ]:
# Demonstrate semantic blindness of BoW
demo_docs = [
    "The game is amazing and fun to play",
    "The game is terrible and boring to play",
    "I love this app it is excellent",
    "I love this app it is great",
]
demo_vec = CountVectorizer()
demo_matrix = demo_vec.fit_transform(demo_docs).toarray()
demo_df = pd.DataFrame(demo_matrix, columns=demo_vec.get_feature_names_out(),
                       index=[f'Doc{i+1}' for i in range(4)])
print('BoW vectors — notice Doc1 & Doc2 differ only in amazing/terrible:')
print(demo_df.to_string())
print('\n→ BoW cannot tell Doc1 (positive) from Doc2 (negative) without the sentiment words')

### Q2 — When to use BoW vs TF-IDF in industry

| Scenario | Best Choice | Reason |
|----------|-------------|--------|
| Short text classification (spam, sentiment) | BoW | Simple, fast, works well |
| Document search / information retrieval | **TF-IDF** | Highlights distinctive keywords |
| Topic modelling (LDA) | BoW | LDA expects raw counts |
| Keyword extraction | **TF-IDF** | IDF naturally ranks important words |
| Feature engineering baseline | Both | Compare and pick the better one |

### Q3 — Limitations of TF-IDF

1. **No semantics** — still treats words as independent; "happy" and "joyful" are unrelated  
2. **Context-blind** — "bank" (river) vs "bank" (finance) get the same score  
3. **Corpus-dependent** — IDF scores change if you add/remove documents  
4. **Sparse & high-dimensional** — same sparsity problem as BoW  
5. **No word order** — "not good" and "good" treated differently only by token, not meaning

---
## Task 7: Mini Use Case — Sentiment Classification

In [ ]:
# Create sentiment labels
# Google Play: rating 1-2 = negative, 4-5 = positive (skip 3 = neutral)
# Steam: 'positive' / 'negative' already in rating column

def get_sentiment(row):
    if row['source'] == 'google_play':
        try:
            r = int(row['rating'])
            if r <= 2: return 'negative'
            if r >= 4: return 'positive'
        except:
            pass
        return None
    elif row['source'] == 'steam':
        return row['rating'] if row['rating'] in ('positive', 'negative') else None
    return None

df['sentiment'] = df.apply(get_sentiment, axis=1)
df_labeled = df.dropna(subset=['sentiment']).copy()

print('Labeled dataset shape:', df_labeled.shape)
print(df_labeled['sentiment'].value_counts())

In [ ]:
X = df_labeled['clean_text']
y = df_labeled['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# Vectorize
bow_vec   = CountVectorizer(max_features=5000)
tfidf_vec = TfidfVectorizer(max_features=5000)

X_train_bow   = bow_vec.fit_transform(X_train)
X_test_bow    = bow_vec.transform(X_test)

X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)

In [ ]:
results = []

for feat_name, X_tr, X_te in [('BoW', X_train_bow, X_test_bow),
                                ('TF-IDF', X_train_tfidf, X_test_tfidf)]:
    for clf_name, clf in [('Logistic Regression', LogisticRegression(max_iter=1000)),
                           ('Naive Bayes',         MultinomialNB())]:
        clf.fit(X_tr, y_train)
        y_pred = clf.predict(X_te)
        acc = accuracy_score(y_test, y_pred)
        results.append({'Features': feat_name, 'Classifier': clf_name, 'Accuracy': round(acc, 4)})
        print(f'{feat_name} + {clf_name}')
        print(classification_report(y_test, y_pred))
        print('-' * 50)

In [ ]:
# Summary table
results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print('\n=== Performance Summary ===')
print(results_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(results_df))
bars = ax.bar(x, results_df['Accuracy'], color=['#2196F3','#FF5722','#4CAF50','#9C27B0'])
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r['Features']}\n{r['Classifier']}" for _, r in results_df.iterrows()],
    fontsize=9
)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Sentiment Classification: BoW vs TF-IDF')
for bar, val in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.2%}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## Observations & Conclusions

1. **Preprocessing** significantly reduces vocabulary size and noise — lemmatization groups inflected forms, stopword removal removes uninformative tokens.

2. **Vocabulary** — after preprocessing the corpus has ~5,000–10,000 unique meaningful words from 1,000 reviews.

3. **OHE** is the weakest representation — loses all frequency information; useful only when word presence/absence matters.

4. **BoW** captures word frequency and works well for short-text classification. It is simple and fast.

5. **TF-IDF** outperforms BoW in tasks requiring keyword importance because it down-weights ubiquitous words ("good", "app") and up-weights distinctive ones.

6. **Sparsity** is a key challenge — ~99% of matrix values are zero. Sparse storage (CSR format) reduces memory by 100×+ compared to dense arrays.

7. **Sentiment classification**: TF-IDF + Logistic Regression typically achieves the best accuracy; Naive Bayes is competitive and faster to train.